# CatBoost на 48 стартах. Пики для h3–h8
Контроль воспроизводит конфигурацию stepa.ipynb с Public 0,331. В Kaggle включите GPU и подключите исходные train_solo_track.parquet и test_solo_track.parquet. Запустите все ячейки по порядку. Нужны 14 оценочных обучений и восемь финальных. Каждый горизонт обрабатывается отдельно.

Кандидат добавляет к контролю три признака. Для каждой из семи предыдущих дат берём объёмы в целевом слоте и в соседних получасах. Усредняем дневные максимумы, размахи и разности правого и левого слота. Горизонты h1 и h2 служат частью полного прогноза при каждом сравнении.


In [ ]:
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
from IPython.display import display

def input_file(name):
    local = Path("data") / name
    if local.is_file():
        return local
    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise ValueError(f"Укажите путь к {name} вручную. Найдено: {matches}")
    return matches[0]

TRAIN_PATH = input_file("train_solo_track.parquet")
TEST_PATH = input_file("test_solo_track.parquet")
OUT = Path("artifacts/stepa_peaks/features")
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "target_1h"
STATUSES = [f"status_{i}" for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = "stepa_peaks_v1"
PEAKS = ["peak7_max", "peak7_range", "peak7_right_minus_left"]

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=["route_id", "timestamp", TARGET, *STATUSES]).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(TEST_PATH, columns=["id", "route_id", "timestamp"])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history
if not all(np.isfinite(array).all() for array in values.values()):
    raise ValueError("В числовых рядах есть бесконечные значения")


dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError("Недостаточно дат для временного разбиения")
tune_start, holdout_start = dates[-28], dates[-14]
LAG_VALIDATION_FIT_BEFORE = tune_start.normalize()

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

means = {(name, steps): rolling(array, steps)
         for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + "_hourly"] = values[name] + past(values[name], positions - 1)
extra_signals["warehouse_total_load"] = sum(extra_signals[name + "_hourly"] for name in STATUSES[:3])
extra_signals["warehouse_processing_rate"] = (
    extra_signals["status_3_hourly"] / (extra_signals["status_1_hourly"] + np.float32(1e-5)))
for name in STATUSES[3:]:
    extra_signals[name + "_diff_hourly"] = values[name] - past(values[name], positions - 2)
extra_signals["hub_buffer"] = values["status_5"] - values["status_4"]
extra_signals["hub_gate_bottleneck"] = (
    extra_signals["status_4_diff_hourly"] - extra_signals["status_6_diff_hourly"])
extra_signals["target_acceleration"] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals["status_6_diff_hourly"]

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            if name in STATUSES[:3]:
                common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f"target_1h_lag_{lag}"] = past(target, origins - lag)
    common["warehouse_total_load_lag_48"] = past(extra_signals["warehouse_total_load"], origins - 48)
    common["status_6_diff_hourly_lag_48"] = past(hub_signal, origins - 48)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(
        recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

CORRELATION_COLUMNS = ["selected_status6_lag_steps", "selected_status6_correlation",
                       "selected_status6_pair_count", "status6_lag_valid",
                       "lagged_status6_signal", "lagged_status6_signal_available"]
FEATURE_GROUPS = {
    "calendar": ["hour", "minute", "hour_float", "hour_sin", "hour_cos", "dayofweek",
                 "is_weekend", "weekday_sin", "weekday_cos"],
    "local_statuses": ["status_1_hourly", "status_2_hourly", "status_3_hourly",
                       "warehouse_total_load", "warehouse_processing_rate"],
    "hub_combinations": ["status_4_diff_hourly", "status_5_diff_hourly", "status_6_diff_hourly",
                         "hub_buffer", "hub_gate_bottleneck"],
    "historical_lags": ["target_1h_lag_48", "target_1h_lag_96",
                        "warehouse_total_load_lag_48", "status_6_diff_hourly_lag_48"],
    "target_change": ["target_acceleration"],
    "correlation_lag": CORRELATION_COLUMNS,
}

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *,
                               fit_before, max_lookback_hours=24,
                               min_pairs=336, min_correlation=0.05, route_block_size=128):

    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = np.asarray(target_values), np.asarray(status6_diff)
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError("Ожидаются два массива [время, маршрут] одной формы")
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError("Для выбора лага нужна упорядоченная получасовая сетка без дубликатов")
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError("Идентификаторы маршрутов повторяются")
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or not steps.is_integer():
        raise ValueError("Граница поиска должна задавать целое число получасов, не меньше двух")
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError("min_pairs должен быть целым числом >=3")
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError("Некорректные параметры выбора корреляции")
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side="left"))

    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = np.where(valid, x, 0), np.where(valid, z, 0)
            divisor = np.maximum(count, 1)
            sx, sz = xx.sum(axis=0), zz.sum(axis=0)
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish-start, np.nan),
                             where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({"route_id": route_ids,
                         "selected_status6_lag_steps": np.where(accepted, best_lag, np.nan),
                         "selected_status6_correlation": best_corr,
                         "selected_status6_pair_count": best_pairs,
                         "status6_lag_valid": accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200_000):

    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError("Некорректный горизонт или размер блока")
    by_route = lag_table.set_index("route_id").reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError("Строка признаков не сопоставлена с историей")
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = (np.isfinite(selected) & (selected >= h) & (source >= 0)
                     & (source <= origin) & (source < len(times)))
        signal = hub_signal[np.clip(source, 0, len(times)-1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions["lagged_status6_signal"]] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions["lagged_status6_signal_available"]] = available.astype(np.float32)
    return frame

validation_lag_table = calculate_dynamic_shoulder(
    target, hub_signal, times, routes, fit_before=LAG_VALIDATION_FIT_BEFORE,
    max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS,
    min_correlation=LAG_MIN_CORRELATION)
validation_lag_table.to_csv(OUT.parent / "status6_lags_validation.csv", index=False)
(OUT.parent / "feature_pipeline.json").write_text(json.dumps({
    "version": FEATURE_PIPELINE_VERSION,
    "validation_lag_fit_before": str(LAG_VALIDATION_FIT_BEFORE),
    "max_lookback_hours": LAG_MAX_LOOKBACK_HOURS, "min_pairs": LAG_MIN_PAIRS,
    "min_correlation": LAG_MIN_CORRELATION,
    "correlation_groups": FEATURE_GROUPS,
}, ensure_ascii=False, indent=2))

In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {
        "hour": dates.hour, "minute": dates.minute,
        "hour_float": dates.hour + dates.minute / 60.0,
        "dayofweek": dates.dayofweek, "is_weekend": (dates.dayofweek >= 5).astype(np.int8),
    }
    calendar["hour_sin"] = np.sin(2 * np.pi * calendar["hour_float"] / 24)
    calendar["hour_cos"] = np.cos(2 * np.pi * calendar["hour_float"] / 24)
    calendar["weekday_sin"] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar["weekday_cos"] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = past(target, slot - days*48).ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d*48) for d in days]).ravel()
    if h >= 3:
        daily = {name: [] for name in PEAKS}
        for day in range(1, 8):
            center = slot - day * 48
            left, mid, right = (past(target, center + k) for k in (-1, 0, 1))
            maximum = np.maximum(np.maximum(left, mid), right)
            minimum = np.minimum(np.minimum(left, mid), right)
            daily[PEAKS[0]].append(maximum)
            daily[PEAKS[1]].append(maximum - minimum)
            daily[PEAKS[2]].append(right - left)
        for name, arrays in daily.items():
            frame[name] = average_available(arrays).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day*48)
        adjusted = past(target, slot - day*48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    numeric = frame.columns.difference(["cutoff", "timestamp", "route_id"])
    return frame.astype({name: "float32" for name in numeric})

def write_horizon(h):
    stop = len(times) - h
    path = OUT / f"history_h{h}.parquet"
    temporary = path.with_suffix(".partial")
    writer = None
    try:
        for start in range(0, stop, 48):
            origins = np.arange(start, min(start + 48, stop))
            frame = horizon_dataset(h, origins)
            frame["y"] = target[origins + h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h*STEP)]
    future = requests.merge(features, on=["route_id", "timestamp"], how="left", validate="one_to_one")
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError("Ошибка сопоставления тестовых строк")
    future.to_parquet(OUT / f"submission_h{h}.parquet", index=False)
    print(f"h={h}: история {stop * len(routes):,}, тест {len(future):,}")

## Хронологическая оценка
Все обучающие метки предшествуют полуночи первого дня подбора. Следующие 14 стартов в 10:30 служат для early stopping и выбора горизонтов. Последние 14 стартов используются для локального test. Таблицы каждого fit содержат WAPE, смещение, их конкурсную сумму и среднюю дневную метрику.


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count

if get_gpu_device_count() < 1:
    raise RuntimeError("Для обучения включите GPU в настройках Kaggle Notebook.")
MODEL_DIR = OUT.parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
print("Начало подбора:", tune_start, "Начало отложенной проверки:", holdout_start)

def make_model(trees):
    return CatBoostRegressor(iterations=trees, depth=7, learning_rate=0.05,
                             loss_function="MAE", eval_metric="MAE", task_type="GPU",
                             devices="0", cat_features=["route_id"], random_seed=42,
                             allow_writing_files=False, verbose=100)

def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({"cutoff": frame.cutoff.dt.normalize().to_numpy(), "volume": actual,
                           "error": errors, "absolute": np.abs(errors)}).groupby("cutoff").sum()

    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {"выборка": split, "даты": frame.cutoff.dt.normalize().nunique(), "строки": len(frame),
            "WAPE": wape, "Relative Bias": signed_bias, "|Relative Bias|": abs(signed_bias),
            "метрика конкурса": wape + abs(signed_bias),
            "среднее по дням": mean_daily_score(frame, predictions)}

TARGET_SCALE = 100_000

def model_columns(data):
    return data.columns.drop(["timestamp", "cutoff", "y", "profile28", "overlap"]).tolist()

split_dates = {"train": dates[dates < tune_start],
               "подбор": dates[(dates >= tune_start) & (dates < holdout_start)],
               "test": dates[dates >= holdout_start]}
display(pd.DataFrame([{"выборка": name, "с": period.min(), "по": period.max(), "даты": len(period)}
                      for name, period in split_dates.items()]))

In [ ]:
validation_parts = []
tree_counts = {}
reports = {}
for h in range(1, 9):
    write_horizon(h)
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    train = data.timestamp < tune_start.normalize()
    tune = data.cutoff.isin(split_dates["подбор"])
    evaluate = data.cutoff.isin(dates[-28:])
    meta = data.loc[evaluate, ["cutoff", "timestamp", "route_id", "y", "profile28"]].copy()
    meta["horizon"] = h
    for variant in (["control", "peaks"] if h >= 3 else ["control"]):
        columns = [c for c in model_columns(data) if variant == "peaks" or c not in PEAKS]
        model = make_model(1500)
        model.fit(data.loc[train, columns], data.loc[train, "y"] / TARGET_SCALE,
                  eval_set=(data.loc[tune, columns], data.loc[tune, "y"] / TARGET_SCALE),
                  early_stopping_rounds=100, use_best_model=True)
        tree_counts[h, variant] = model.tree_count_
        meta[variant] = np.maximum(model.predict(data.loc[evaluate, columns]) * TARGET_SCALE, 0)
        train_pred = np.maximum(model.predict(data.loc[train, columns]) * TARGET_SCALE, 0)
        test_rows = meta.loc[meta.cutoff >= holdout_start]
        table = pd.DataFrame([score_row(data.loc[train], train_pred, "train"),
                              score_row(test_rows, test_rows[variant], "test")])
        reports[h, variant] = table.assign(horizon=h, model=variant, trees=model.tree_count_)
        display(reports[h, variant])
        del model, train_pred
        gc.collect()
    validation_parts.append(meta)
    del data
    gc.collect()
validation = pd.concat(validation_parts, ignore_index=True)
validation.to_parquet(OUT.parent / "validation_predictions.parquet", index=False)
pd.concat(reports.values()).to_csv(OUT.parent / "fit_metrics.csv", index=False)

## Выбор на подборе
Используем постобработку сохранённого прогона 0,331. На h1 берём CatBoost, на h2–h8 добавляем 30% profile28. Общую аддитивную поправку оцениваем на подборе по средней дневной метрике. При сравнении кандидатов поправка одна и та же. Идём от h3 к h8 и принимаем замену только при строгом улучшении полного прогноза с уже выбранными горизонтами.


In [ ]:
from scipy.optimize import minimize_scalar

def raw_prediction(frame, selected):
    model_pred = frame.control.to_numpy(dtype=float).copy()
    mask = frame.horizon.isin(selected)
    if mask.any():
        model_pred[mask] = frame.loc[mask, "peaks"]
    raw = np.where(frame.horizon == 1, model_pred, 0.7 * model_pred + 0.3 * frame.profile28)
    if not np.isfinite(raw).all():
        raise ValueError("Неконечный прогноз")
    return raw

def calibrate(frame, selected):
    raw = raw_prediction(frame, selected)
    limit = 2 * max(float(frame.y.mean()), 1)
    def objective(offset):
        return mean_daily_score(frame, np.maximum(0, raw + offset))
    grid = np.linspace(-limit, limit, 65)
    scores = [objective(x) for x in grid]
    best = int(np.argmin(scores))
    low, high = grid[max(best - 1, 0)], grid[min(best + 1, 64)]
    fit = minimize_scalar(objective, bounds=(low, high), method="bounded")
    return float(min([0, grid[best], fit.x, np.mean(frame.y - raw)], key=objective))

def forecast(frame, selected, offset):
    return np.maximum(0, raw_prediction(frame, selected) + offset)

tuning = validation.loc[validation.cutoff < holdout_start]
holdout = validation.loc[validation.cutoff >= holdout_start]
offset = calibrate(tuning, [])
selected = []
score = mean_daily_score(tuning, forecast(tuning, selected, offset))
decisions = []
for h in range(3, 9):
    trial = selected + [h]
    candidate = mean_daily_score(tuning, forecast(tuning, trial, offset))
    decisions.append(dict(horizon=h, previous=score, candidate=candidate, accepted=candidate < score))
    if candidate < score:
        selected, score = trial, candidate
display(pd.DataFrame(decisions))
pd.DataFrame(decisions).to_csv(OUT.parent / "tuning_decisions.csv", index=False)
rows = [score_row(frame, forecast(frame, horizons, offset), period) | {"model": name}
        for period, frame in [("подбор", tuning), ("test", holdout)]
        for name, horizons in [("stepa", []), ("selected", selected)]]
report = pd.DataFrame(rows)
display(report)
report.to_csv(OUT.parent / "comparison.csv", index=False)
print("Оценка на локальном test получена независимо от финального переобучения и до него. Эти даты ранее просматривались.")

## Финальное переобучение
Для восьми выбранных моделей переоцениваем лаги и обучаем CatBoost на всех стартах с доступной целевой меткой. Выбор горизонтов зафиксирован. Для конкурсного прогноза общую поправку пересчитываем по всем 28 дням исторических прогнозов.


In [ ]:
final_lag_table = calculate_dynamic_shoulder(
    target, hub_signal, times, routes, fit_before=end + STEP,
    max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS,
    min_correlation=LAG_MIN_CORRELATION)
final_lag_table.to_csv(OUT.parent / "status6_lags_final.csv", index=False)
metadata_path = OUT.parent / "feature_pipeline.json"
metadata = json.loads(metadata_path.read_text())
metadata.update(final_lag_fit_before=str(end + STEP), target_scale=TARGET_SCALE)
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
submission_parts = []
final_offset = calibrate(validation, selected)
for h in range(1, 9):
    variant = "peaks" if h in selected else "control"
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    future = pd.read_parquet(OUT / f"submission_h{h}.parquet")
    data = add_correlation_features(data, h, final_lag_table)
    future = add_correlation_features(future, h, final_lag_table)
    columns = [c for c in model_columns(data) if variant == "peaks" or c not in PEAKS]
    model = make_model(tree_counts[h, variant])
    model.fit(data[columns], data.y / TARGET_SCALE)
    model.save_model(str(MODEL_DIR / f"h{h}_{variant}.cbm"))
    full_pred = np.maximum(model.predict(data[columns]) * TARGET_SCALE, 0)
    full_row = score_row(data, full_pred, "вся история после refit")
    display(pd.concat([reports[h, variant], pd.DataFrame([dict(full_row, horizon=h, model=variant, trees=model.tree_count_)])]))
    result = future[["id", "profile28"]].copy()
    result["horizon"] = h
    result["control"] = np.maximum(model.predict(future[columns]) * TARGET_SCALE, 0)
    result["y_pred"] = forecast(result, [], final_offset)
    submission_parts.append(result[["id", "y_pred"]])
    del model, data, full_pred
    gc.collect()

predictions = pd.concat(submission_parts, ignore_index=True)
submission = test[["id"]].merge(predictions, on="id", how="outer", sort=False, validate="one_to_one")
submission = submission.set_index("id").reindex(test.id).reset_index()
if len(predictions) != len(test) or not np.isfinite(submission.y_pred).all() or (submission.y_pred < 0).any():
    raise ValueError("Нарушена полнота или допустимость прогнозов")
submission.to_csv("submission.csv", index=False)
settings = dict(selected_horizons=selected, tuning_offset=offset, submission_offset=final_offset)
(OUT.parent / "settings.json").write_text(json.dumps(settings, indent=2))
print("submission.csv", submission.shape, settings)
print("Независимая от refit оценка локального test получена до переобучения всех восьми моделей.")